# E044 — MAP Relevance Factor r Ablation (Tied Covariance)

**Motivation:** E013 tested MAP r for diagonal UBM (r=16 optimal). Tied covariance may have different optimal r due to different parameter count and regularization.

**Hypothesis:** Tied covariance UBM may prefer different MAP relevance factor than diagonal (r=16).

**Configs:** r ∈ {4, 8, 16, 32, 64}

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

import numpy as np
import librosa
from scipy.special import logsumexp
from sklearn.mixture import GaussianMixture
import copy

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

SEED = 67
UBM_COMPONENTS = 32

DATA = Path('../data').resolve()
manifest = load_manifest(DATA)
y_all = manifest['label'].to_numpy()
print(f'{len(manifest)} samples')

E042_REF = {'mean_eer': 0.46, 'std_eer': 0.65}

222 samples


In [2]:
def _find_wav(stem, data_dir):
    for sf in ("target_train", "target_dev", "non_target_train", "non_target_dev"):
        p = data_dir / sf / (stem + ".wav")
        if p.exists(): return p
    raise FileNotFoundError(stem)

def _extract_lpcc(y, sr, order=12, n_cep=13, hop_length=160, win_length=400):
    frames = librosa.util.frame(y, frame_length=win_length, hop_length=hop_length)
    cep_frames = []
    for frame in frames.T:
        frame = frame * np.hanning(len(frame))
        try:
            a = librosa.lpc(frame.astype(np.float64), order=order)
            A_freq = np.fft.rfft(a, n=512)
            log_H = -np.log(np.abs(A_freq) + 1e-10)
            cep = np.real(np.fft.irfft(log_H))[:n_cep]
        except Exception:
            cep = np.zeros(n_cep)
        cep_frames.append(cep)
    feat = np.array(cep_frames, dtype=np.float32)
    delta = librosa.feature.delta(feat.T).T
    delta2 = librosa.feature.delta(feat.T, order=2).T
    feat = np.hstack([feat, delta, delta2])
    feat -= feat.mean(axis=0)
    return feat

def _aug_pitch(y, sr, rng):
    n_steps = float(rng.choice([-2, -1, 1, 2]))
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)

def train_tied_with_r(train_df, data_dir, MAP_R, seed):
    rng = np.random.default_rng(seed)
    X_target, X_nontarget = [], []
    for _, row in train_df.iterrows():
        y_wav, sr = librosa.load(_find_wav(row["stem"], data_dir), sr=None, mono=True)
        wavs = [y_wav]
        if row["label"] == 1:
            wavs.append(_aug_pitch(y_wav, sr, rng))
        for y_aug in wavs:
            feat = _extract_lpcc(y_aug, sr)
            if row["label"] == 1:
                X_target.append(feat)
            else:
                X_nontarget.append(feat)
    ubm = GaussianMixture(n_components=UBM_COMPONENTS, covariance_type='tied', max_iter=200, random_state=SEED).fit(np.vstack(X_nontarget))
    log_resp = ubm._estimate_log_prob(np.vstack(X_target)) + np.log(ubm.weights_)
    log_resp -= logsumexp(log_resp, axis=1, keepdims=True)
    resp = np.exp(log_resp)
    n_k = resp.sum(axis=0)
    mu_hat = (resp.T @ np.vstack(X_target)) / (n_k[:, None] + 1e-10)
    alpha = n_k / (n_k + MAP_R)
    adapted = copy.deepcopy(ubm)
    adapted.means_ = alpha[:, None] * mu_hat + (1 - alpha[:, None]) * ubm.means_
    return ubm, adapted

def score_with_tta(wav_path, adapted, ubm):
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    scores = []
    for rate in [0.9, 1.0, 1.1]:
        y_stretched = librosa.effects.time_stretch(y, rate=rate)
        feat = _extract_lpcc(y_stretched, sr)
        scores.append(adapted.score(feat) - ubm.score(feat))
    return np.mean(scores)

print('Functions defined')

Functions defined


## 2. Cross-validation

In [3]:
r_values = [4, 8, 16, 32, 64]
results = {}

for r in r_values:
    print(f"\n=== r={r} ===")
    fold_eers = []
    for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
        seed_f = SEED + fold_id
        train_df = manifest.loc[train_idx]
        val_df = manifest.loc[val_idx]
        ubm, adapted = train_tied_with_r(train_df, DATA, r, seed_f)
        scores = [score_with_tta(_find_wav(row["stem"], DATA), adapted, ubm) for _, row in val_df.iterrows()]
        scores = np.array(scores)
        eer, _ = compute_eer(scores[y_all[val_idx] == 1], scores[y_all[val_idx] == 0])
        fold_eers.append(eer * 100)
        print(f"  Fold {fold_id}: EER={eer*100:.2f}%")
    results[r] = {'fold_eers': fold_eers, 'mean': np.mean(fold_eers), 'std': np.std(fold_eers)}
    print(f"  Mean +/- Std: {np.mean(fold_eers):.2f} +/- {np.std(fold_eers):.2f}%")

print("\n=== Summary ===")
for r, res in results.items():
    print(f"r={r:2d}: {res['mean']:5.2f} +/- {res['std']:5.2f}%")


=== r=4 ===


  Fold 0: EER=1.39%


  Fold 1: EER=0.00%


  Fold 2: EER=0.00%
  Mean +/- Std: 0.46 +/- 0.65%

=== r=8 ===


  Fold 0: EER=1.39%


...

  Fold 1: EER=0.00%


  Fold 2: EER=0.00%
  Mean +/- Std: 0.46 +/- 0.65%

=== r=64 ===


  Fold 0: EER=1.39%


  Fold 1: EER=0.00%


  Fold 2: EER=0.00%
  Mean +/- Std: 0.46 +/- 0.65%

=== Summary ===
r= 4:  0.46 +/-  0.65%
r= 8:  0.46 +/-  0.65%
r=16:  0.46 +/-  0.65%
r=32:  0.46 +/-  0.65%
r=64:  0.46 +/-  0.65%


## 3. Conclusion

Optimal r for tied cov: [TBD]
Decision: [Keep r=16 or adopt new value]